# 10.8 沒附圖時，包裝會改變原文字計算嗎？


同一份文字模型、同一串三個文字編號，分別走原入口與加了圖片功能的包裝入口。沒有附圖時，希望兩者的候選分數逐項相同；這是介面接通前後的計算核對，不是問模型是否會寫文章。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.model import TinyLM, ModelConfig
from tiny_perceptron.multimodal import MultiModalLM

torch.manual_seed(0)
lm = TinyLM(ModelConfig(width=8)).eval()
multimodal = MultiModalLM(lm).eval()
ids = torch.tensor([1, 20, 30])
a = lm(ids[None])["logits"]
b = multimodal(ids)["logits"]
print("分數形狀", tuple(a.shape))
print("純文字入口相同", torch.equal(a, b))

`MultiModalLM(lm)` 包住的是同一個文字模型，不是另一份新權重。兩邊都設為 `eval()`，本例避開訓練模式下的隨機行為。原入口以 `ids[None]` 補批次軸，包裝則在內部處理它。

三個編號沒有圖片標記，輸出都是 `(1,3,264)`：一筆、三位置、264 個文字候選。`torch.equal` 印 True，確認包裝沿用相同文字計算。即使權重隨機、兩路都答不好，一致性仍可以成立；它和答案正確是兩個問題。

如果放圖片標記卻沒有圖片，應報缺資料。嘗試 `multimodal(torch.tensor([1,5,4]))` 會出現佔位標記缺少配對圖片／聲音的錯誤，不能默默把它當普通字詞。

訓練後若開放文字權重，計算可能合理地改變；是否保住舊技能，要用固定文字題核對答案。若改包裝後立即不同，先查輸入、位置與運算路徑；若只在訓練後退步，再查更新範圍與資料。練習換成 `[1,40,50]`，預期形狀與一致性仍相同。

<details>
<summary>回顧與查證</summary>

可回顧：[10.7模態標記與展開](https://birdhackor.github.io/tiny-perceptron-vlm/10.7.html)、[4.6輸出候選分數](https://birdhackor.github.io/tiny-perceptron-vlm/4.6.html)、[16.8](https://birdhackor.github.io/tiny-perceptron-vlm/16.8.html)。

</details>
